# L15 · Agentic RL: tool을 쓰며 여러 turn 학습하기

## Goal

- single response와 step MDP를 구분한다
- tool output을 policy loss에서 mask한다
- outcome과 process credit을 비교한다

## Setup

이 cell은 CPU·seed·offline 상태와 split hash를 먼저 고정합니다. toy 연산은 결정론적인 CPU 연산만 쓰며, package trainer의 전역 결정론 기본값은 유지합니다.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L15:toy:42").hexdigest()
print(f"lesson=L15 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L15 language=ko profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.12.13 rl_study=0.1.0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:6991fdc13d01bf81da402c446d2cd63648a4265912685088c353e1e91edc7ebb data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. 현재 위치와 핵심 식

⏱ 5분 · 1/3 section · [필수/CORE]

현재 위치: LLM policy → **Agentic RL multi-turn MDP** → trajectory 평가

$$G_t=r_t^{process}+\sum_{k=t}^{T}\gamma^{k-t}r_k^{outcome}$$

agent는 observation에서 `CALL` 또는 `FINAL` action을 만들고 tool output을 다음 observation으로 받습니다. policy가 생성한 action token만 loss 대상이며 tool output은 context일 뿐입니다. outcome reward broadcast와 discounted process return은 서로 다른 credit 가정입니다.

### 2. 작은 숫자로 실행

⏱ 6분 · 2/3 section · [필수/CORE]

**먼저 예측:** tool output text를 다음 context에 넣을 때 그 token도 현재 policy loss mask에 포함할까요? 20초 동안 답을 적은 뒤 실행하세요.

<details><summary>정답 보기</summary>아니요. 환경이 만든 token을 policy action처럼 학습하면 gradient ownership이 깨집니다.</details>

In [2]:
from rl_study.agentic.envs import CalculatorToolEnv
from rl_study.agentic.trajectory import rollout_episode, update_policy
from rl_study.models import TinyCausalLM, TinyLMConfig, TinyTokenizer
agent_model = TinyCausalLM(TinyLMConfig(
    max_sequence_length=192, hidden_size=32, num_heads=4,
    num_layers=1, intermediate_size=64
))
agent_tokenizer = TinyTokenizer()
agent_trajectory, generated, rollout_forwards = rollout_episode(
    agent_model, agent_tokenizer, CalculatorToolEnv(seed=42, max_steps=3),
    generator=torch.Generator().manual_seed(42), policy_version=0, task_index=0
)
agent_optimizer = torch.optim.AdamW(agent_model.parameters(), lr=1e-3)
agent_update = update_policy(
    agent_model, agent_tokenizer, agent_optimizer, agent_trajectory,
    current_policy_version=0, credit_mode="discounted_returns", gamma=0.95
)
print({"episode_steps": len(agent_trajectory.steps), "generated_tokens": generated,
       "rollout_forwards": rollout_forwards, "credits": agent_update.step_credits,
       "tool_outputs_masked": True, "loss": round(agent_update.loss, 4)})

{'episode_steps': 1, 'generated_tokens': 14, 'rollout_forwards': 1, 'credits': (-0.25,), 'tool_outputs_masked': True, 'loss': -0.2648}


### 3. 구현 해부

⏱ 6분 · 3/3 section · [심화/DEEP DIVE]

**왜 이렇게 구현했나:** rollout 당시 token ID와 candidate set, policy version을 trajectory에 보존해 retokenization과 stale-policy drift를 탐지합니다. text만 저장하는 단순 log는 사람이 읽기 쉽지만 정확한 update 재현에는 부족합니다.

**흔한 함정:** outcome reward를 모든 step에 넣고 discounted return에서도 다시 더하면 reward를 이중 계산합니다. outcome은 종료 step에 한 번 저장하고 credit 함수가 배분합니다. 회귀 test: `test_rollout_preserves_original_action_tokens_and_masks_tool_output`.

**쉬어가기:** 지금 출력한 한 값만 설명할 수 있으면 다음 cell로 가세요.

## Checks

In [3]:
assert len(agent_trajectory.steps) >= 1
assert all(step.action_token_ids for step in agent_trajectory.steps)
print("checks=passed")

checks=passed


**회상 문제:** process reward와 outcome reward를 같은 scalar로 미리 합치면 어떤 진단 능력을 잃나요? 1~2문장으로 답하세요.

## 내가 자주 틀리는 것

- loss가 유한하면 구현도 맞다고 생각한다.
- `terminated`와 `truncated`, prompt와 action을 합친다.
- 한 seed의 작은 결과를 알고리즘 순위로 확대한다.

## 60초 요약

- **실행 결론:** 이번 seeded rollout은 한 step에서 종료되어 credit -0.25를 받았고 update loss는 유한했습니다. 실패 rollout도 mask와 credit 계약을 검증하는 유효한 증거입니다.
- 실제 확인: `test_rollout_preserves_original_action_tokens_and_masks_tool_output`.
- 출력은 고정 seed의 toy 실행이며 논문 규모 결과가 아닙니다.

## Next Steps

1. L16에서 success 하나가 아니라 reward hacking, budget, split, initial hash까지 묶어 실험을 감사합니다.
2. `[필수/CORE]` assertion을 한 번 깨뜨리고 오류를 읽습니다.
3. package test를 열어 notebook의 작은 식과 production guard를 연결합니다.

[상세 구현 문서](../../docs/algorithms/agentic-rl.md) · [강좌 지도](../../docs/course-map.md)

## Sources

- `agent-lightning-2025` — `docs/sources.yml`
- `agent-r1-2025` — `docs/sources.yml`
- `framework-agent-lightning` — `docs/sources.yml`
- `repo-agent-r1` — `docs/sources.yml`
- `benchmark-alfworld` — `docs/sources.yml`